Generate deterministic mechanism-fit validation receipt

In [1]:
cd /workspace/backend && PYTHONPATH=. python scripts/validate_mechanism_fit_retriever.py /mnt/shared-workspace/shared/phase_d_sprint/w2/mechanism_fit_validation_receipt.json

{
  "receipt": "/mnt/shared-workspace/shared/phase_d_sprint/w2/mechanism_fit_validation_receipt.json",
  "sha256": "8694fc8eae96e074f9d7e35b1e1ccb71c53431adbfe19cd02f2589917223f8f5",
  "deterministic": true,
  "forbidden": [],
  "states": {
    "MBD4_TP53_HGSOC": "SUPPORTED_CONTEXT",
    "HGSOC_BRCA1": "SUPPORTED_CONTEXT",
    "MSIH_OVARIAN": "SUPPORTED_CONTEXT",
    "MSIH_CRC": "SUPPORTED_CONTEXT",
    "STC1010_CRC_TRACE": "SUPPORTED_CONTEXT",
    "WRONG_LINEAGE": "OUT_OF_SCOPE",
    "CONTRADICTED": "CONTRADICTED_CONTEXT",
    "MISSING_MECHANISM": "INSUFFICIENT_EVIDENCE"
  }
}


Audit live Project Data Sphere ovarian packages

In [3]:
from pathlib import Path
import requests, json, hashlib, os, re
from datetime import datetime, timezone

OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane2_pds'); OUT.mkdir(parents=True,exist_ok=True)
DATA=Path('/workspace/mechanism_fit_pds'); DATA.mkdir(parents=True,exist_ok=True)
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def sha_bytes(b): return hashlib.sha256(b).hexdigest()
def save(name,obj):
    p=OUT/name; p.write_text(json.dumps(obj,indent=2,default=str)); return p

def sanitized_response(r,url,at):
    return {'url':url,'timestamp':at,'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':r.text[:5000]}

base='https://data.projectdatasphere.org'
s=requests.Session(); manifest={'lane':'PDS ovarian packages','started_at':ts(),'secret_presence':{'PDS_USERNAME':bool(os.getenv('PDS_USERNAME') or os.getenv('PDS_EMAIL')),'PDS_PASSWORD':bool(os.getenv('PDS_PASSWORD'))},'catalog':{},'ovarian_packages':[]}
# Public filters and full catalog.
for method,path,payload,name in [('GET','/api/accessdata/filters',None,'filters'),('POST','/api/accessdata/results',{},'catalog')]:
    url=base+path; at=ts()
    try:
        r=s.request(method,url,json=payload,timeout=90)
        rec=sanitized_response(r,url,at); save(f'{name}_receipt.json',rec)
        if r.status_code==200:
            obj=r.json(); save(f'{name}.json',obj); manifest['catalog'][name]={'status':200,'timestamp':at,'receipt':f'{name}.json'}
            if name=='catalog': catalog=obj
        else: manifest['catalog'][name]={'status':r.status_code,'timestamp':at,'receipt':f'{name}_receipt.json'}
    except Exception as e:
        rec={'url':url,'timestamp':at,'status_code':None,'error':type(e).__name__+': '+str(e)[:500]}; save(f'{name}_receipt.json',rec); manifest['catalog'][name]=rec

# Normalize unknown catalog envelope, then select every record containing ovarian in tumor fields or serialized record.
def records(obj):
    if isinstance(obj,list): return obj
    if isinstance(obj,dict):
        for k in ['results','content','donations','data','items']:
            if isinstance(obj.get(k),list): return obj[k]
        for v in obj.values():
            if isinstance(v,list) and v and isinstance(v[0],dict): return v
    return []
all_records=records(locals().get('catalog',{}))
manifest['catalog']['record_count']=len(all_records)
ovarian=[]
for rec in all_records:
    text=json.dumps(rec,default=str).lower()
    if 'ovarian' in text or 'ovary' in text:
        ovarian.append(rec)
save('ovarian_catalog_records.json',ovarian)

# No configured credentials in this worker: perform exact guest-session and login-readiness receipts without submitting an empty password.
at=ts(); url=base+'/api/public/user/session'
try:
    r=s.get(url,timeout=60); save('guest_session_receipt.json',sanitized_response(r,url,at))
except Exception as e: save('guest_session_receipt.json',{'url':url,'timestamp':at,'error':type(e).__name__+': '+str(e)[:500]})

for rec in ovarian:
    did=rec.get('donationId') or rec.get('id') or rec.get('donation_id')
    package={'catalog_record':rec,'donationId':did,'access_timestamp':ts(),'detail':None,'files':[]}
    if did is None:
        package['error']='No donationId in live catalog record'; manifest['ovarian_packages'].append(package); continue
    url=f'{base}/api/accessdata/results/{did}'; at=ts()
    try:
        r=s.get(url,timeout=90); srec=sanitized_response(r,url,at); save(f'donation_{did}_detail_receipt.json',srec)
        if r.status_code==200:
            detail=r.json(); save(f'donation_{did}_detail.json',detail); package['detail']=detail
            # Traverse for file objects with an id/fileId and name/fileName.
            found=[]
            def walk(x):
                if isinstance(x,dict):
                    if any(k in x for k in ['fileId','fileName','filename']) and any(k in x for k in ['id','fileId','datasetId']): found.append(x)
                    for v in x.values(): walk(v)
                elif isinstance(x,list):
                    for v in x: walk(v)
            walk(detail)
            seen=set()
            for f in found:
                fid=f.get('fileId') or f.get('id'); key=(str(fid),str(f.get('fileName') or f.get('filename') or f.get('name')))
                if key in seen: continue
                seen.add(key)
                fe={'metadata':f,'access_timestamp':ts()}
                if fid:
                    durl=f'{base}/api/file/download/{did}/{fid}'; dat=ts()
                    rr=s.get(durl,timeout=90,allow_redirects=False)
                    drec=sanitized_response(rr,durl,dat); save(f'donation_{did}_file_{fid}_access_receipt.json',drec)
                    fe.update({'url':durl,'status_code':rr.status_code,'receipt':f'donation_{did}_file_{fid}_access_receipt.json'})
                    # Only retain a body if the guest endpoint truly returns a non-HTML file.
                    ctype=rr.headers.get('content-type','').lower()
                    if rr.status_code==200 and 'text/html' not in ctype and len(rr.content)>0:
                        name=f.get('fileName') or f.get('filename') or f.get('name') or f'{did}_{fid}.bin'
                        dst=DATA/(f'{did}_{fid}_'+Path(str(name)).name); dst.write_bytes(rr.content)
                        fe.update({'local_path':str(dst),'bytes':dst.stat().st_size,'sha256':hashlib.sha256(rr.content).hexdigest(),'access':'ACCESSED'})
                    else: fe['access']='ACCESS_DENIED'
                package['files'].append(fe)
        else: package['detail_status']=r.status_code
    except Exception as e:
        package['error']=type(e).__name__+': '+str(e)[:500]
    manifest['ovarian_packages'].append(package)

manifest['completed_at']=ts(); save('lane2_manifest.json',manifest)
print(json.dumps({'catalog_records':len(all_records),'ovarian_packages':len(ovarian),'file_access_attempts':sum(len(p['files']) for p in manifest['ovarian_packages']),'credential_pair_present':all(manifest['secret_presence'].values()),'output':str(OUT)},indent=2))

{
  "catalog_records": 99,
  "ovarian_packages": 1,
  "file_access_attempts": 6,
  "credential_pair_present": false,
  "output": "/mnt/shared-workspace/shared/mechanism_fit_audit/lane2_pds"
}


Cross-review ovarian outcome benchmark eligibility

In [5]:
from pathlib import Path
import json,hashlib
from datetime import datetime,timezone
root=Path('/mnt/shared-workspace/shared/mechanism_fit_audit'); src=root/'lane4_geo'
a=json.loads((src/'GSE30161_matrix_and_complete_case_audit.json').read_text())
b=json.loads((src/'GSE63885_matrix_and_complete_case_audit.json').read_text())
review={'reviewer_lane':'PDS worker cross-review','timestamp':datetime.now(timezone.utc).isoformat().replace('+00:00','Z'),'sources':['GSE30161','GSE63885'],'findings':[
 {'source':'GSE30161','molecular_matrix_accessed':a['access']['status_code']==200,'sample_count':a['sample_count'],'explicit_pfi_event_usable_count':54,'treatment_context':'adjuvant platinum-based chemotherapy','treatment_start_or_systemic_index_date_present':False,'verdict':'ACCESSIBLE_ENDPOINT_INCOMPATIBLE','exact_missing_field':'Exact systemic-treatment start/index date is absent; surgery/diagnosis dates do not prove chemotherapy index date.'},
 {'source':'GSE63885','molecular_matrix_accessed':b['access']['status_code']==200,'sample_count':b['sample_count'],'explicit_clinical_complete_count':b['explicit_complete_count'],'treatment_context':'first-line adjuvant chemotherapy','treatment_start_or_systemic_index_date_present':False,'verdict':'ACCESSIBLE_ENDPOINT_INCOMPATIBLE','exact_missing_field':'Exact systemic-treatment start/index date and endpoint-specific censoring date are absent.'}],
 'decision':'No source meets the approved outcome gate requiring pretreatment input plus treatment line and index date.'}
p=root/'cross_review_outcome.json';p.write_text(json.dumps(review,indent=2));print(json.dumps(review,indent=2))

{
  "reviewer_lane": "PDS worker cross-review",
  "timestamp": "2026-07-16T21:04:56.601901Z",
  "sources": [
    "GSE30161",
    "GSE63885"
  ],
  "findings": [
    {
      "source": "GSE30161",
      "molecular_matrix_accessed": true,
      "sample_count": 58,
      "explicit_pfi_event_usable_count": 54,
      "treatment_context": "adjuvant platinum-based chemotherapy",
      "treatment_start_or_systemic_index_date_present": false,
      "verdict": "ACCESSIBLE_ENDPOINT_INCOMPATIBLE",
      "exact_missing_field": "Exact systemic-treatment start/index date is absent; surgery/diagnosis dates do not prove chemotherapy index date."
    },
    {
      "source": "GSE63885",
      "molecular_matrix_accessed": true,
      "sample_count": 101,
      "explicit_clinical_complete_count": 75,
      "treatment_context": "first-line adjuvant chemotherapy",
      "treatment_start_or_systemic_index_date_present": false,
      "verdict": "ACCESSIBLE_ENDPOINT_INCOMPATIBLE",
      "exact_missing_field": "